# Spatiotemporal Compliance Assessment System (SCAS)

End-to-end pipeline for emergency vehicle compliance assessment: dataset download, K-Fold cross-validation training of a YOLOv8-cls + ResNet18 ensemble, DeepSORT tracking, spatiotemporal compliance checking, and SQLite violation logging.


## 1. Environment Setup

Install dependencies and import all required libraries. Uncomment the `!pip install` line when running on Google Colab.


In [1]:
# Uncomment the next line if running on Google Colab
!pip install ultralytics deep-sort-realtime kagglehub torch torchvision opencv-python matplotlib seaborn scikit-learn streamlit --quiet

import os
import shutil
import random
import json
import math
import sqlite3
from pathlib import Path
from collections import defaultdict
from datetime import datetime

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import models

import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from ultralytics import YOLO

# Device configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Using device: {device}")

# Global constants
CLASS_NAMES = ["non-emergency", "emergency"]  # 0=non-emergency, 1=emergency
EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

print("[✓] Environment setup complete")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 117.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 118.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[INFO] Using device: cuda
[✓] Environment setup complete


## 2. Output Directory Setup

Resolves all output paths relative to `Capstone/outputs/` regardless of where Jupyter was launched and creates all required subdirectories on first run.


In [ ]:
# ── Output Paths ──────────────────────────────────────────────────────────────

_here         = Path().resolve()
_ROOT         = _here.parent if _here.name == "notebooks" else _here
_OUTPUTS      = _ROOT / "outputs"

VIDEO_OUT_DIR = _OUTPUTS / "videos"
EVIDENCE_DIR  = _OUTPUTS / "evidence_frames"
DB_DEMO_PATH  = _OUTPUTS / "database" / "violations_demo.db"
PLOTS_DIR     = _OUTPUTS / "plots"

for _d in [VIDEO_OUT_DIR, EVIDENCE_DIR, DB_DEMO_PATH.parent, PLOTS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

print(f"[INFO] All outputs -> {_OUTPUTS}")
print(f"[INFO]   videos   : {VIDEO_OUT_DIR}")
print(f"[INFO]   evidence : {EVIDENCE_DIR}")
print(f"[INFO]   database : {DB_DEMO_PATH.parent}")
print(f"[INFO]   plots    : {PLOTS_DIR}")

[INFO] All outputs -> /content/outputs
[INFO]   videos   : /content/outputs/videos
[INFO]   evidence : /content/outputs/evidence_frames
[INFO]   database : /content/outputs/database
[INFO]   plots    : /content/outputs/plots


## 3. Dataset Acquisition

Downloads the emergency vs non-emergency vehicle classification dataset from Kaggle, then pools all images from `train/`, `validation/`, and `test/` splits into flat lists for K-Fold processing.


In [3]:
def download_dataset() -> str:
    """
    Downloads the emergency vs non-emergency vehicle classification
    dataset from Kaggle using kagglehub.

    Returns:
        str: Local path to the downloaded dataset
    """
    import kagglehub
    path = kagglehub.dataset_download(
        "parthplc/emergency-vs-nonemergency-vehicle-classification"
    )
    print(f"[INFO] Dataset downloaded to: {path}")
    return path


def find_root(dataset_path: str) -> Path:
    """
    Locates the root folder that contains train/validation/test subfolders.
    Handles cases where Kaggle adds an extra wrapper folder on download.

    Args:
        dataset_path: Path returned by download_dataset()

    Returns:
        Path: The root directory containing train/val/test folders

    Raises:
        FileNotFoundError: If train/val/test structure not found
    """
    p = Path(dataset_path)

    # Check current directory and all immediate subdirectories
    for candidate in [p, *p.iterdir()]:
        if candidate.is_dir() and any(
            (candidate / s).is_dir() for s in ("train", "validation", "test")
        ):
            return candidate

    raise FileNotFoundError(
        f"Cannot find train/validation/test structure under {dataset_path}"
    )


def collect_all_samples(dataset_path: str) -> tuple:
    """
    Pools all images from every split (train/validation/test) into
    one flat list. We do this to manage our own K-Fold splits rather
    than using the pre-defined ones.

    Why pool everything?
    - Pre-defined splits may be imbalanced
    - K-Fold stratification ensures consistent class distribution
    - Allows us to control train/val/test proportions

    Args:
        dataset_path: Path to the dataset root

    Returns:
        tuple: (image_paths, labels)
            - image_paths: List of absolute file paths (strings)
            - labels: Parallel list of integers (0 or 1)
    """
    root = find_root(dataset_path)
    image_paths = []
    labels = []

    # Iterate through train/validation/test folders
    for split_dir in root.iterdir():
        if not split_dir.is_dir():
            continue

        # Each split has folders "0" and "1"
        for cls_folder in sorted(split_dir.iterdir()):
            if not cls_folder.is_dir() or cls_folder.name not in ("0", "1"):
                continue

            label = int(cls_folder.name)  # 0=non-emergency, 1=emergency

            for img_path in cls_folder.iterdir():
                if img_path.suffix.lower() in EXTS:
                    image_paths.append(str(img_path))
                    labels.append(label)

    print(f"[INFO] Total images collected: {len(image_paths)}")
    print(f"[INFO]   Non-emergency (0): {labels.count(0)}")
    print(f"[INFO]   Emergency (1):     {labels.count(1)}")

    return image_paths, labels

print("[✓] Dataset functions defined")

[✓] Dataset functions defined


## 4. K-Fold Cross-Validation Splits

Creates K stratified folds with a permanently held-out test set (fold K). For each CV fold `i`: val = fold `i`, train = all remaining CV folds, test = fixed held-out fold.


In [ ]:
def make_kfold_splits(image_paths: list, labels: list,
                      k: int = 5, seed: int = 42) -> list:
    """
    Divides all images into K stratified folds with held-out test set.

    Args:
        image_paths: List of image file paths
        labels: Parallel list of labels (0 or 1)
        k: Number of folds (default 5)
        seed: Random seed for reproducibility

    Returns:
        list: List of dicts, one per CV fold, each containing:
            - fold: Fold index (0 to k-2)
            - train: List of training indices
            - val: List of validation indices
            - test: List of test indices (same for all folds)
    """
    # Create stratified K-Fold splitter
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)

    # Generate all fold indices
    folds = [idx.tolist() for _, idx in skf.split(image_paths, labels)]

    # Last fold is permanently held out as test set
    test_idx = folds[-1]
    cv_folds = folds[:-1]  # Remaining K-1 folds for cross-validation

    splits = []
    for i, val_fold in enumerate(cv_folds):
        # Training = all CV folds except the current validation fold
        train_folds = [f for j, f in enumerate(cv_folds) if j != i]
        train_idx = [idx for fold in train_folds for idx in fold]  # Flatten list

        splits.append({
            "fold": i,
            "train": train_idx,
            "val": val_fold,
            "test": test_idx,
        })

    # Print summary
    print(f"\n[INFO] K-Fold split configuration (K={k}):")
    print(f"[INFO]   Held-out test set: {len(test_idx)} images (fold {k-1})")
    for s in splits:
        print(f"[INFO]   CV fold {s['fold']} → "
              f"train={len(s['train'])}, val={len(s['val'])}")

    return splits

def write_fold_dataset(image_paths: list, labels: list,
                       split_info: dict, fold_dir: str) -> str:
    """
    Physically copies images into YOLOv8-cls expected folder structure.

    YOLOv8-cls requires this structure:
        fold_dir/
          train/
            non-emergency/  ← Class 0 training images
            emergency/      ← Class 1 training images
          val/
            non-emergency/
            emergency/
          test/
            non-emergency/
            emergency/

    Args:
        image_paths: List of all image paths
        labels: Parallel list of labels
        split_info: Dict from make_kfold_splits() containing train/val/test indices
        fold_dir: Output directory for this fold

    Returns:
        str: Path to fold_dir (used as data= in YOLO training)
    """
    fold_dir = Path(fold_dir)

    # Clean up previous fold data to avoid stale images
    if fold_dir.exists():
        shutil.rmtree(fold_dir)

    # Create directory structure and copy images
    for split_name, indices in [
        ("train", split_info["train"]),
        ("val", split_info["val"]),
        ("test", split_info["test"])
    ]:
        # Create class subfolders
        for cls_name in CLASS_NAMES:
            (fold_dir / split_name / cls_name).mkdir(parents=True, exist_ok=True)

        # Copy images to correct class folder
        for idx in indices:
            src = Path(image_paths[idx])
            cls_name = CLASS_NAMES[labels[idx]]

            # Prefix with index to avoid filename collisions across splits
            dst = fold_dir / split_name / cls_name / f"{idx}_{src.name}"
            shutil.copy2(src, dst)

    return str(fold_dir)

print("[✓] K-Fold split functions defined")

[✓] K-Fold split functions defined


## 5. PyTorch Dataset & Augmentation

Defines training transforms (random crop, flip, colour jitter, rotation) and validation transforms (deterministic centre crop), plus the custom `VehicleDataset` class for ResNet18.


In [5]:
# Training transforms — WITH augmentation
TRAIN_TF = T.Compose([
    T.Resize((256, 256)),          # Slightly larger than target
    T.RandomCrop(224),             # Random 224×224 crop
    T.RandomHorizontalFlip(),      # Random left-right flip (50% chance)
    T.ColorJitter(                 # Random color variations
        brightness=0.3,            # ±30% brightness
        contrast=0.3,              # ±30% contrast
        saturation=0.2             # ±20% saturation
    ),
    T.RandomRotation(15),          # Random rotation ±15 degrees
    T.ToTensor(),                  # Convert PIL Image to PyTorch tensor [0,1]
    T.Normalize(                   # Normalize to ImageNet statistics
        [0.485, 0.456, 0.406],     # Mean (R, G, B)
        [0.229, 0.224, 0.225]      # Std (R, G, B)
    ),
])

# Validation/Test transforms — NO augmentation
VAL_TF = T.Compose([
    T.Resize((224, 224)),          # Direct resize (no crop)
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class FoldDataset(Dataset):
    """
    PyTorch Dataset for loading images from a fold directory.

    Automatically applies training or validation transforms based on split name.
    Shuffles samples to ensure mixed batches during training.

    Args:
        fold_dir: Path to fold directory (created by write_fold_dataset)
        split: "train", "val", or "test"
    """

    def __init__(self, fold_dir: str, split: str):
        # Choose transforms based on split
        self.tf = TRAIN_TF if split == "train" else VAL_TF
        self.samples = []  # List of (image_path, label) tuples

        split_dir = Path(fold_dir) / split

        # Collect all images from class folders
        for cls_name in CLASS_NAMES:
            cls_dir = split_dir / cls_name
            if not cls_dir.is_dir():
                continue

            label = CLASS_NAMES.index(cls_name)  # 0 or 1

            for img_path in cls_dir.iterdir():
                if img_path.suffix.lower() in EXTS:
                    self.samples.append((str(img_path), label))

        # Shuffle to mix classes in batches
        random.shuffle(self.samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        """
        Loads and transforms one image.

        Returns:
            tuple: (transformed_image_tensor, label)
        """
        img_path, label = self.samples[idx]

        # Open as RGB (ensures 3 channels even if grayscale)
        img = Image.open(img_path).convert("RGB")

        return self.tf(img), label

print("[✓] Data augmentation and Dataset class defined")

[✓] Data augmentation and Dataset class defined


## 6. YOLOv8-cls Fine-Tuning

Fine-tunes `yolov8n-cls.pt` (ImageNet pretrained) on a single CV fold for 8 epochs with early stopping (patience = 10).


In [6]:
def train_yolo_cls_fold(fold_dir: str, fold_idx: int,
                        epochs: int = 8, imgsz: int = 224,
                        batch: int = 32) -> str:
    """
    Fine-tunes YOLOv8n-cls on one CV fold.

    Args:
        fold_dir: Path to fold directory with train/val/test splits
        fold_idx: Current fold index (for naming)
        epochs: Number of training epochs
        imgsz: Input image size (224×224)
        batch: Batch size (reduce if GPU memory errors)

    Returns:
        str: Path to best.pt weights file
    """
    print(f"\n{'='*60}")
    print(f"  Training YOLOv8-cls on Fold {fold_idx}")
    print(f"{'='*60}")

    # Load pretrained YOLOv8 nano classification model
    model = YOLO("yolov8n-cls.pt")

    # Train
    results = model.train(
        data=fold_dir,           # Path to fold directory
        epochs=epochs,           # Number of epochs
        imgsz=imgsz,             # Input size (224)
        batch=batch,             # Batch size
        name=f"emerg_cls_fold{fold_idx}",  # Experiment name
        device=0 if torch.cuda.is_available() else "cpu",
        patience=10,             # Early stopping patience
        augment=True,            # Enable built-in augmentation
        plots=True,              # Save training plots
    )

    # Get path to best weights
    best_weights = str(Path(results.save_dir) / "weights" / "best.pt")
    print(f"[✓] YOLOv8-cls fold {fold_idx} best weights: {best_weights}")

    return best_weights

print("[✓] YOLOv8 training function defined")

[✓] YOLOv8 training function defined


## 7. ResNet18 Architecture

Builds ResNet18 with a custom binary head: backbone → GlobalAvgPool → Dropout(0.4) → Linear(512 → 2). Pretrained ImageNet weights provide a feature extractor complementary to YOLOv8.


In [7]:
def build_resnet(pretrained: bool = True) -> nn.Module:
    """
    Builds ResNet18 with custom classification head.

    Args:
        pretrained: If True, loads ImageNet pretrained weights

    Returns:
        nn.Module: ResNet18 model ready for training
    """
    # Load ResNet18 with or without pretrained weights
    model = models.resnet18(
        weights=models.ResNet18_Weights.DEFAULT if pretrained else None
    )

    # Replace final fully connected layer
    # Original: Linear(512 → 1000) for ImageNet classes
    # New: Linear(512 → 2) for emergency classification
    in_features = model.fc.in_features  # 512

    model.fc = nn.Sequential(
        nn.Dropout(0.4),                        # Dropout for regularization
        nn.Linear(in_features, len(CLASS_NAMES)),  # Binary classification
    )

    return model.to(device)

print("[✓] ResNet18 architecture defined")

[✓] ResNet18 architecture defined


## 8. ResNet18 Training

Trains ResNet18 with differential learning rates (backbone `lr×0.1`, head `lr`), label smoothing (0.1), AdamW optimiser, and cosine-annealing scheduler.


In [8]:
def train_resnet_fold(fold_dir: str, fold_idx: int,
                      epochs: int = 4, batch: int = 32,
                      lr: float = 1e-4) -> tuple:
    """
    Trains ResNet18 on one CV fold.

    Args:
        fold_dir: Path to fold directory
        fold_idx: Current fold index
        epochs: Number of training epochs (4 is usually sufficient)
        batch: Batch size
        lr: Learning rate for new classification head

    Returns:
        tuple: (path_to_weights, best_validation_accuracy)
    """
    print(f"\n{'='*60}")
    print(f"  Training ResNet18 on Fold {fold_idx}")
    print(f"{'='*60}")

    # Create datasets and dataloaders
    train_ds = FoldDataset(fold_dir, "train")
    val_ds = FoldDataset(fold_dir, "val")

    train_dl = DataLoader(
        train_ds, batch_size=batch, shuffle=True,
        num_workers=2, pin_memory=True
    )
    val_dl = DataLoader(
        val_ds, batch_size=batch, shuffle=False,
        num_workers=2, pin_memory=True
    )

    # Build model
    model = build_resnet(pretrained=True)

    # Loss function with label smoothing
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # Optimizer with differential learning rates
    # Backbone: lr × 0.1, Head: lr
    optimizer = optim.AdamW([
        {"params": list(model.parameters())[:-2], "lr": lr * 0.1},  # Backbone
        {"params": list(model.parameters())[-2:], "lr": lr},        # Head
    ], weight_decay=1e-4)

    # Learning rate scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # Training loop
    best_acc = 0.0
    save_path = f"resnet18_fold{fold_idx}.pt"

    for epoch in range(1, epochs + 1):
        # --- TRAINING PHASE ---
        model.train()
        train_loss = 0.0

        for imgs, lbls in train_dl:
            imgs, lbls = imgs.to(device), lbls.to(device)

            # Forward pass
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, lbls)

            # Backward pass
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        scheduler.step()

        # --- VALIDATION PHASE ---
        model.eval()
        correct = total = 0

        with torch.no_grad():
            for imgs, lbls in val_dl:
                imgs, lbls = imgs.to(device), lbls.to(device)
                preds = model(imgs).argmax(dim=1)
                correct += (preds == lbls).sum().item()
                total += len(lbls)

        val_acc = correct / total

        # Save best model
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), save_path)

        # Print progress every 2 epochs
        if epoch % 2 == 0 or epoch == epochs:
            print(f"    Epoch {epoch:3d}/{epochs}  "
                  f"val_acc={val_acc:.4f}  best={best_acc:.4f}")

    print(f"[✓] ResNet18 fold {fold_idx} — best val acc: {best_acc:.4f}")
    return save_path, best_acc

print("[✓] ResNet18 training function defined")

[✓] ResNet18 training function defined


## 9. K-Fold Training Runner

Orchestrates the full K-Fold loop: trains YOLOv8-cls and ResNet18 on each fold, records ensemble validation accuracy, selects the best fold, and saves `cv_summary.json`.


In [9]:
def run_kfold_cv(image_paths: list, labels: list,
                 k: int = 5, yolo_epochs: int = 8,
                 resnet_epochs: int = 4, batch: int = 32) -> dict:
    """
    Runs full K-Fold cross-validation loop.

    Args:
        image_paths: List of all image paths
        labels: Parallel list of labels
        k: Number of folds (default 5)
        yolo_epochs: Epochs for YOLOv8 training
        resnet_epochs: Epochs for ResNet training
        batch: Batch size

    Returns:
        dict: CV results containing:
            - cv_results: List of per-fold results
            - best: Best fold info (weights, accuracies)
            - mean_acc: Mean validation accuracy across folds
            - std_acc: Standard deviation
    """
    # Create K-Fold splits
    splits = make_kfold_splits(image_paths, labels, k=k)

    cv_results = []

    # Train on each CV fold
    for split_info in splits:
        fold_idx = split_info["fold"]

        print(f"\n{'='*60}")
        print(f"  CV FOLD {fold_idx + 1} / {k - 1}")
        print(f"{'='*60}")

        # Write fold dataset to disk
        fold_dir = f"kfold_data/fold_{fold_idx}"
        write_fold_dataset(image_paths, labels, split_info, fold_dir)

        # Train YOLOv8-cls
        yolo_weights = train_yolo_cls_fold(
            fold_dir, fold_idx, epochs=yolo_epochs, batch=batch
        )

        # Train ResNet18
        resnet_weights, resnet_acc = train_resnet_fold(
            fold_dir, fold_idx, epochs=resnet_epochs, batch=batch
        )

        # Get YOLO validation accuracy
        yolo_model = YOLO(yolo_weights)
        yolo_val = yolo_model.val(data=fold_dir, split="val", verbose=False)
        yolo_acc = float(yolo_val.top1)

        # Compute ensemble accuracy (simple average)
        ensemble_acc = (yolo_acc + resnet_acc) / 2

        print(f"\n  Fold {fold_idx} Results:")
        print(f"    YOLO val acc:    {yolo_acc:.4f}")
        print(f"    ResNet val acc:  {resnet_acc:.4f}")
        print(f"    Ensemble acc:    {ensemble_acc:.4f}")

        # Store results
        cv_results.append({
            "fold": fold_idx,
            "fold_dir": fold_dir,
            "yolo_weights": yolo_weights,
            "resnet_weights": resnet_weights,
            "yolo_val_acc": yolo_acc,
            "resnet_val_acc": resnet_acc,
            "ensemble_val_acc": ensemble_acc,
            "test_indices": split_info["test"],
        })

    # Compute statistics
    accs = [r["ensemble_val_acc"] for r in cv_results]
    mean_acc = np.mean(accs)
    std_acc = np.std(accs)

    # Print summary
    print(f"\n{'='*60}")
    print(f"  K-FOLD CV SUMMARY  (K={k}, CV folds={k-1})")
    print(f"{'='*60}")
    for r in cv_results:
        print(f"  Fold {r['fold']}  ensemble_val_acc = {r['ensemble_val_acc']:.4f}")
    print(f"\n  Mean = {mean_acc:.4f}  ±  {std_acc:.4f}")

    # Plot results
    plt.figure(figsize=(8, 4))
    plt.bar(
        [f"Fold {r['fold']}" for r in cv_results],
        accs,
        color="steelblue",
        edgecolor="black"
    )
    plt.axhline(mean_acc, color="red", linestyle="--",
                label=f"Mean = {mean_acc:.4f}")
    plt.ylim(0, 1)
    plt.ylabel("Ensemble Val Accuracy")
    plt.title(f"K-Fold CV Results  (K={k})")
    plt.legend()
    plt.tight_layout()
    plt.savefig("kfold_cv_results.png", dpi=130)
    plt.show()
    print("[✓] Saved: kfold_cv_results.png")

    # Select best fold
    best = max(cv_results, key=lambda r: r["ensemble_val_acc"])
    print(f"\n[✓] Best fold: {best['fold']} "
          f"(ensemble_val_acc={best['ensemble_val_acc']:.4f})")

    return {
        "cv_results": cv_results,
        "best": best,
        "mean_acc": float(mean_acc),
        "std_acc": float(std_acc),
    }

print("[✓] K-Fold CV runner defined")

[✓] K-Fold CV runner defined


## 10. Ensemble Classifier

Combines YOLOv8-cls and ResNet18 via probability averaging with 5-view test-time augmentation (TTA) on ResNet. Predictions below the confidence threshold (0.70) default to non-emergency.


In [10]:
class EnsembleClassifier:
    """
    Ensemble classifier combining YOLOv8-cls and ResNet18 predictions.

    Args:
        yolo_cls_weights: Path to trained YOLOv8-cls weights
        resnet_weights: Path to trained ResNet18 weights
        yolo_weight: Weight for YOLO predictions (default 0.5)
        resnet_weight: Weight for ResNet predictions (default 0.5)
        conf_threshold: Minimum confidence for emergency classification
    """

    # TTA transforms (class-level constants)
    _NORM = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    TTA_TRANSFORMS = [
        T.Compose([T.Resize((224, 224)), T.ToTensor(), _NORM]),
        T.Compose([T.Resize((224, 224)), T.RandomHorizontalFlip(1), T.ToTensor(), _NORM]),
        T.Compose([T.Resize((256, 256)), T.CenterCrop(224), T.ToTensor(), _NORM]),
        T.Compose([T.Resize((224, 224)), T.ColorJitter(0.2, 0.2), T.ToTensor(), _NORM]),
        T.Compose([T.Resize((224, 224)), T.RandomRotation(10), T.ToTensor(), _NORM]),
    ]

    def __init__(self, yolo_cls_weights: str, resnet_weights: str,
                 yolo_weight: float = 0.5, resnet_weight: float = 0.5,
                 conf_threshold: float = 0.7):

        # Load YOLO model
        self.yolo_cls = YOLO(yolo_cls_weights)

        # Load ResNet model
        self.resnet = build_resnet(pretrained=False)
        self.resnet.load_state_dict(
            torch.load(resnet_weights, map_location=device)
        )
        self.resnet.to(device)
        self.resnet.eval()

        # Ensemble weights
        self.yw = yolo_weight
        self.rw = resnet_weight
        self.conf_thresh = conf_threshold

    def _resnet_predict_tta(self, pil_img: Image.Image) -> np.ndarray:
        """
        Runs ResNet18 with Test-Time Augmentation.

        Averages predictions across 5 augmented versions for robustness.

        Args:
            pil_img: PIL Image (RGB)

        Returns:
            np.ndarray: Averaged probability vector [P(non-emergency), P(emergency)]
        """
        self.resnet.eval()
        probs = []

        with torch.no_grad():
            for tf in self.TTA_TRANSFORMS:
                # Transform and add batch dimension
                tensor = tf(pil_img).unsqueeze(0).to(device)

                # Forward pass
                logits = self.resnet(tensor)
                prob = torch.softmax(logits, dim=1).cpu().numpy()[0]
                probs.append(prob)

        # Average across all augmentations
        return np.mean(probs, axis=0)

    def predict(self, pil_img: Image.Image) -> tuple:
        """
        Predicts emergency vs non-emergency using ensemble.

        Args:
            pil_img: PIL Image of vehicle crop

        Returns:
            tuple: (label, confidence, combined_probabilities)
                - label: "emergency" or "non-emergency"
                - confidence: float in [0, 1]
                - combined: np.ndarray of ensemble probabilities
        """
        # --- YOLO PREDICTION ---
        yolo_result = self.yolo_cls(pil_img, verbose=False)[0]
        yolo_probs = yolo_result.probs.data.cpu().numpy()

        # --- RESNET PREDICTION ---
        resnet_probs = self._resnet_predict_tta(pil_img)

        # --- ALIGN YOLO OUTPUT TO CLASS_NAMES ---
        # YOLOv8-cls output order might not match CLASS_NAMES
        yolo_names = self.yolo_cls.names  # {0: 'non-emergency', 1: 'emergency'}
        reordered_yolo_probs = np.zeros_like(yolo_probs)

        for yolo_idx, yolo_name in yolo_names.items():
            if yolo_name in CLASS_NAMES:
                target_idx = CLASS_NAMES.index(yolo_name)
                reordered_yolo_probs[target_idx] = yolo_probs[yolo_idx]

        yolo_probs = reordered_yolo_probs

        # --- ENSEMBLE ---
        combined = self.yw * yolo_probs + self.rw * resnet_probs

        # --- FINAL PREDICTION ---
        cls_id = int(np.argmax(combined))
        conf = float(combined[cls_id])
        label = CLASS_NAMES[cls_id]

        # --- CONFIDENCE THRESHOLD ---
        if conf < self.conf_thresh:
            # Default to non-emergency if uncertain
            label = "non-emergency"
            conf = float(combined[CLASS_NAMES.index("non-emergency")])

        return label, conf, combined

print("[✓] Ensemble classifier defined")

[✓] Ensemble classifier defined


## 11. Held-Out Test Set Evaluation

Evaluates the best-fold ensemble on fold K — data never seen during training or fold selection. Outputs a classification report, confusion matrix, and auto-detects label inversion.


In [11]:
def evaluate_on_test_fold(cv_info: dict, image_paths: list, labels: list):
    """
    Evaluates best fold on held-out test set.

    Args:
        cv_info: Results dict from run_kfold_cv()
        image_paths: List of all image paths
        labels: Parallel list of labels
    """
    best = cv_info["best"]
    test_indices = best["test_indices"]

    print(f"\n{'='*60}")
    print(f"  HELD-OUT TEST SET EVALUATION")
    print(f"  Using best fold: {best['fold']}")
    print(f"  Val accuracy: {best['ensemble_val_acc']:.4f}")
    print(f"{'='*60}")

    # Instantiate ensemble classifier
    clf = EnsembleClassifier(
        yolo_cls_weights=best["yolo_weights"],
        resnet_weights=best["resnet_weights"],
        yolo_weight=0.5,
        resnet_weight=0.5,
        conf_threshold=0.7
    )

    preds, trues = [], []

    print(f"[INFO] Evaluating on {len(test_indices)} test images...")

    # Run predictions on test set
    for idx in test_indices:
        try:
            pil = Image.open(image_paths[idx]).convert("RGB")
            pred, _, _ = clf.predict(pil)

            # Map prediction to integer
            pred_int = 1 if pred == "emergency" else 0
            preds.append(pred_int)
            trues.append(labels[idx])

        except Exception as e:
            print(f"  [WARNING] Skipping index {idx}: {e}")

    if not preds:
        print("[ERROR] No predictions generated — check dataset paths")
        return

    # --- DIAGNOSTIC: CHECK FOR LABEL INVERSION ---
    orig_acc = sum(p == t for p, t in zip(preds, trues)) / len(trues)
    flip_acc = sum((1 - p) == t for p, t in zip(preds, trues)) / len(trues)

    print(f"\n[DEBUG] Original accuracy: {orig_acc:.4f}")
    print(f"[DEBUG] Flipped accuracy:  {flip_acc:.4f}")

    # Auto-fix if flipped is better
    if flip_acc > orig_acc:
        print("[INFO] Detected label inversion. Auto-correcting...")
        preds = [1 - p for p in preds]

    # --- CLASSIFICATION REPORT ---
    print("\n" + "="*60)
    print("  CLASSIFICATION REPORT")
    print("="*60)
    print(classification_report(
        trues, preds,
        target_names=CLASS_NAMES,
        labels=list(range(len(CLASS_NAMES)))
    ))

    # --- CONFUSION MATRIX ---
    cm = confusion_matrix(trues, preds, labels=list(range(len(CLASS_NAMES))))

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Reds",
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES
    )
    plt.title(
        f"Confusion Matrix — Held-Out Test Fold\n"
        f"(Best fold: {best['fold']}, val_acc={best['ensemble_val_acc']:.4f})"
    )
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig("confusion_matrix_test.png", dpi=130)
    plt.show()

    print("[✓] Saved: confusion_matrix_test.png")

print("[✓] Test evaluation function defined")

[✓] Test evaluation function defined


## 12. Classification Demo

Visual sanity check: runs the ensemble on a sample of test images and displays predicted label, ground-truth, and confidence for each crop.


In [12]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random, torch
from pathlib import Path

def run_classification_demo(cv_info: dict, image_paths: list, labels: list,
                             n_samples: int = 12, seed: int = 42):
    """
    Visually demonstrates the trained ensemble on a sample of test images.
    Displays predicted label, ground-truth, and confidence for each sample.
    """
    yolo_w   = cv_info["best"]["yolo_cls_weights"]
    resnet_w = cv_info["best"]["resnet_weights"]
    clf      = EnsembleClassifier(yolo_w, resnet_w)

    # Sample equally from both classes when possible
    random.seed(seed)
    idx0 = [i for i, l in enumerate(labels) if l == 0]
    idx1 = [i for i, l in enumerate(labels) if l == 1]
    half = n_samples // 2
    sampled = (random.sample(idx0, min(half, len(idx0))) +
               random.sample(idx1, min(half, len(idx1))))
    random.shuffle(sampled)

    cols = 4
    rows = (len(sampled) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
    axes = axes.flatten()

    correct = 0
    for ax, i in zip(axes, sampled):
        img_path = image_paths[i]
        gt_label = CLASS_NAMES[labels[i]]
        try:
            from PIL import Image
            img = Image.open(img_path).convert("RGB")
            pred_label, conf, _ = clf.predict(img)
        except Exception as e:
            ax.axis('off')
            ax.set_title(f"Error\n{e}", fontsize=8, color='red')
            continue

        is_correct = pred_label == gt_label
        if is_correct:
            correct += 1
        border_color = '#10b981' if is_correct else '#e8272b'

        ax.imshow(img)
        ax.axis('off')
        for spine in ax.spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(3)
            spine.set_visible(True)
        ax.set_title(
            f"Pred: {pred_label} ({conf:.0%})\nTrue: {gt_label}",
            fontsize=9,
            color='green' if is_correct else 'red',
        )

    for ax in axes[len(sampled):]:
        ax.axis('off')

    acc = correct / len(sampled) if sampled else 0
    fig.suptitle(
        f"Classification Demo — {correct}/{len(sampled)} correct ({acc:.0%})\n"
        f"Green border = correct  |  Red border = wrong",
        fontsize=12, y=1.01,
    )
    plt.tight_layout()
    plt.savefig(str(PLOTS_DIR / "classification_demo.png"), dpi=120, bbox_inches='tight')
    plt.show()
    print(f"[INFO] Demo accuracy: {acc:.1%}  ({correct}/{len(sampled)} correct)")
    print(f"[INFO] Saved → {PLOTS_DIR / 'classification_demo.png'}")

# ── Run demo (requires cv_info from run_kfold_cv, and image_paths / labels) ──
# Uncomment after training:
# run_classification_demo(cv_info, image_paths, labels, n_samples=12)

## 13. DeepSORT Tracker

Wraps DeepSORT (MobileNet embedder + Kalman filter) to assign persistent `track_id` values across frames. Emergency classification is sticky — once labelled EV a track is not reclassified. Majority-vote over the last 15 frames stabilises labels against per-frame classifier noise.


In [13]:
class DeepSORTTracker:
    """
    Wrapper around DeepSORT for vehicle tracking.
    FIXED: Emergency classification is STICKY (once EV, always EV)
    """

    def __init__(self, max_age: int = 30):
        from deep_sort_realtime.deepsort_tracker import DeepSort

        self.tracker = DeepSort(
            max_age=max_age,
            embedder='mobilenet',
            half=torch.cuda.is_available(),
            embedder_gpu=torch.cuda.is_available(),
        )

        # Store classification history per track ID
        self.history: dict = defaultdict(list)

        # NEW: Track which vehicles have EVER been emergency
        self.emergency_vehicles = set()

    def update(self, detections: list, frame: np.ndarray) -> list:
        """
        Updates tracker with new detections.
        FIXED: Emergency label is sticky - once EV, always EV.
        """
        # Convert to DeepSORT format
        ds_input = [
            ([x1, y1, x2 - x1, y2 - y1], conf, lbl)
            for x1, y1, x2, y2, conf, lbl in detections
        ]

        # Run tracker
        tracks = self.tracker.update_tracks(ds_input, frame=frame)

        outputs = []
        for tr in tracks:
            if not tr.is_confirmed():
                continue

            tid = tr.track_id
            ltrb = tr.to_ltrb()
            label = tr.get_det_class() or "unknown"

            # Store label in history
            self.history[tid].append(label)

            # ========================================
            # STICKY EMERGENCY LOGIC
            # ========================================
            # If this track was EVER classified as emergency, lock it
            if label == "emergency":
                self.emergency_vehicles.add(tid)

            # If track is in emergency set, force label to emergency
            if tid in self.emergency_vehicles:
                stable_label = "emergency"
            else:
                # Normal majority vote for non-emergency vehicles
                recent = self.history[tid][-15:]
                stable_label = max(set(recent), key=recent.count)

            outputs.append((tid, *ltrb, stable_label))

        return outputs

print("[✓] DeepSORT tracker with STICKY emergency labels defined")

[✓] DeepSORT tracker with STICKY emergency labels defined


## 14. Compliance Engine

Rule-based spatiotemporal engine. For each non-EV within `proximity_r` px of an EV, computes `yield_amount = current_distance − distance_W_frames_ago`. If `yield_amount ≤ yield_d`: Minor violation (not yielding); if negative: Major violation (approaching).


In [14]:
class ComplianceEngine:
    """
    FIXED: Works for BOTH dashcam and overhead cameras.
    """

    def __init__(self, proximity_r: int = 180,
                 reaction_w: int = 30,
                 yield_threshold: int = 40,
                 camera_type: str = "dashcam"):  # NEW

        self.R = proximity_r
        self.W = reaction_w
        self.Y = yield_threshold
        self.camera_type = camera_type

        self.distance_history = defaultdict(list)
        self.flagged_pairs = {}
        self.cooldown = defaultdict(int)

        print(f"[INFO] Compliance Engine initialized:")
        print(f"  Camera type: {self.camera_type}")
        print(f"  Proximity radius: {self.R}px")
        print(f"  Reaction window: {self.W} frames")
        print(f"  Yield threshold: {self.Y}px")

    @staticmethod
    def _centre(x1, y1, x2, y2):
        return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)

    @staticmethod
    def _dist(cx1, cy1, cx2, cy2):
        return math.sqrt((cx1 - cx2) ** 2 + (cy1 - cy2) ** 2)

    def _in_same_lane(self, non_ev_x, ev_x, threshold=80):
        """Check if vehicles are horizontally aligned (same lane)."""
        return abs(non_ev_x - ev_x) < threshold

    def _is_blocking(self, non_ev_x, non_ev_y, ev_x, ev_y):
        """
        Check if non-EV is in blocking position relative to EV.

        FIXED: Camera-aware position checking.
        """
        if self.camera_type == "overhead":
            # Top-down: smaller Y = ahead
            return non_ev_y < ev_y

        elif self.camera_type == "dashcam":
            # Dashcam: distance-based (any close vehicle in same lane)
            # The yield_amount determines if they're truly blocking
            return True

        else:
            return True  # Default: check all vehicles

    def check(self, tracks: list, frame_id: int, frame: np.ndarray) -> list:
        violations = []

        # Separate EVs and non-EVs
        ev_tracks = [
            (tid, x1, y1, x2, y2)
            for tid, x1, y1, x2, y2, label in tracks
            if label == "emergency"
        ]

        non_ev_tracks = [
            (tid, x1, y1, x2, y2)
            for tid, x1, y1, x2, y2, label in tracks
            if label == "non-emergency"
        ]

        # No EV → clear state
        if not ev_tracks:
            self.distance_history.clear()
            self.flagged_pairs.clear()
            self.cooldown.clear()
            return violations

        # Reduce cooldowns
        for key in list(self.cooldown.keys()):
            self.cooldown[key] -= 1
            if self.cooldown[key] <= 0:
                del self.cooldown[key]

        # Check each non-EV
        for non_tid, nx1, ny1, nx2, ny2 in non_ev_tracks:
            ncx, ncy = self._centre(nx1, ny1, nx2, ny2)

            for ev_tid, ex1, ey1, ex2, ey2 in ev_tracks:
                ecx, ecy = self._centre(ex1, ey1, ex2, ey2)

                current_dist = self._dist(ncx, ncy, ecx, ecy)
                pair_key = (non_tid, ev_tid)

                # Too far → clear
                if current_dist > self.R:
                    if pair_key in self.distance_history:
                        del self.distance_history[pair_key]
                    if pair_key in self.flagged_pairs:
                        del self.flagged_pairs[pair_key]
                    continue

                # Not in same lane → ignore
                if not self._in_same_lane(ncx, ecx):
                    continue

                # Not blocking → ignore
                if not self._is_blocking(ncx, ncy, ecx, ecy):
                    continue

                # Track distance history
                history = self.distance_history[pair_key]
                history.append(current_dist)

                if len(history) > self.W:
                    history.pop(0)

                if len(history) < self.W:
                    continue

                # Compute yield
                initial_dist = history[0]
                final_dist = history[-1]
                yield_amount = final_dist - initial_dist

                # Check violation
                is_violation = (yield_amount < self.Y)

                if is_violation:
                    if pair_key in self.cooldown:
                        continue

                    # Determine severity
                    if yield_amount < 0:
                        severity = "BLOCKING (approaching)"
                    elif 0 <= yield_amount < 20:
                        severity = "BLOCKING (stationary)"
                    else:
                        severity = "BLOCKING (slow yield)"

                    violations.append({
                        "vehicle_id": non_tid,
                        "ev_id": ev_tid,
                        "frame_id": frame_id,
                        "distance_px": round(current_dist, 1),
                        "yield_amount": round(yield_amount, 1),
                        "severity": severity,
                        "timestamp": datetime.now().isoformat(),
                        "frame": frame,
                        "bbox": (int(nx1), int(ny1), int(nx2), int(ny2)),
                    })

                    self.cooldown[pair_key] = 60
                    self.flagged_pairs[pair_key] = frame_id

                else:
                    if pair_key in self.flagged_pairs:
                        del self.flagged_pairs[pair_key]

        return violations

print("[✓] Compliance engine with camera-aware blocking defined")

[✓] Compliance engine with camera-aware blocking defined


## 15. Violation Logger

Writes violations to a 3NF SQLite schema (`Vehicle`, `VideoFrame`, `Violation`) and saves JPEG bounding-box crops of each offending vehicle as evidence.


In [15]:
class ViolationLogger:
    """Logs violations to SQLite database and saves evidence frames."""

    def __init__(self, db_path: str = "violations.db",
                 evidence_dir: str = "evidence_frames"):

        self.db_path = db_path
        self.evidence_dir = evidence_dir
        os.makedirs(evidence_dir, exist_ok=True)

        self.conn = sqlite3.connect(db_path, check_same_thread=False)
        self._create_tables()

        print(f"[INFO] Violation database: {db_path}")
        print(f"[INFO] Evidence directory: {evidence_dir}/")

    def _create_tables(self):
        cur = self.conn.cursor()

        cur.execute("""
            CREATE TABLE IF NOT EXISTS Vehicle (
                vehicle_id  INTEGER PRIMARY KEY AUTOINCREMENT,
                track_id    TEXT NOT NULL UNIQUE,
                type        TEXT,
                class       TEXT
            )
        """)

        cur.execute("""
            CREATE TABLE IF NOT EXISTS VideoFrame (
                frame_id   INTEGER PRIMARY KEY,
                timestamp  TEXT NOT NULL,
                source     TEXT
            )
        """)

        cur.execute("""
            CREATE TABLE IF NOT EXISTS Violation (
                violation_id   INTEGER PRIMARY KEY AUTOINCREMENT,
                vehicle_id     INTEGER NOT NULL,
                frame_id       INTEGER NOT NULL,
                distance_px    REAL,
                yield_amount   REAL,
                severity       TEXT,
                evidence_path  TEXT,
                timestamp      TEXT,
                FOREIGN KEY (vehicle_id) REFERENCES Vehicle(vehicle_id),
                FOREIGN KEY (frame_id)   REFERENCES VideoFrame(frame_id)
            )
        """)

        self.conn.commit()

    def log(self, violation: dict):
        cur = self.conn.cursor()

        # Insert vehicle
        cur.execute(
            "INSERT OR IGNORE INTO Vehicle (track_id, type, class) VALUES (?,?,?)",
            (violation["vehicle_id"], "non-emergency", "unknown")
        )
        cur.execute(
            "SELECT vehicle_id FROM Vehicle WHERE track_id = ?",
            (violation["vehicle_id"],)
        )
        vehicle_id = cur.fetchone()[0]

        # Insert frame
        cur.execute(
            "INSERT OR IGNORE INTO VideoFrame (frame_id, timestamp, source) "
            "VALUES (?,?,?)",
            (violation["frame_id"], violation["timestamp"], "video_stream")
        )

        # Save evidence (with error handling)
        evidence_path = self._save_evidence(
            frame=violation["frame"],
            bbox=violation["bbox"],
            vehicle_id=violation["vehicle_id"],
            frame_id=violation["frame_id"],
        )

        # Insert violation
        cur.execute("""
            INSERT INTO Violation
                (vehicle_id, frame_id, distance_px, yield_amount, severity,
                 evidence_path, timestamp)
            VALUES (?,?,?,?,?,?,?)
        """, (
            vehicle_id,
            violation["frame_id"],
            violation["distance_px"],
            violation.get("yield_amount", 0),
            violation["severity"],
            evidence_path,
            violation["timestamp"],
        ))

        self.conn.commit()

        print(
            f"  [VIOLATION] Frame {violation['frame_id']:05d} | "
            f"Track {violation['vehicle_id']} | "
            f"Dist: {violation['distance_px']:.1f}px | "
            f"Yield: {violation.get('yield_amount', 0):.1f}px | "
            f"{violation['severity']}"
        )

    def log_many(self, violations: list, frame: np.ndarray):
        for v in violations:
            self.log(v)

    def _save_evidence(self, frame: np.ndarray, bbox: tuple,
                       vehicle_id: str, frame_id: int) -> str:
        """
        Crops and saves evidence image with ROBUST boundary checking.

        FIXED: Handles edge cases where bbox is partially outside frame.
        """
        x1, y1, x2, y2 = bbox
        h, w = frame.shape[:2]

        # Clamp to image boundaries (BEFORE cropping)
        x1_clamped = max(0, min(x1, w - 1))
        y1_clamped = max(0, min(y1, h - 1))
        x2_clamped = max(0, min(x2, w))
        y2_clamped = max(0, min(y2, h))

        # Ensure valid crop dimensions
        if x2_clamped <= x1_clamped or y2_clamped <= y1_clamped:
            # Invalid bbox - create a placeholder
            print(f"  [WARNING] Invalid bbox for track {vehicle_id} frame {frame_id} - skipping evidence")
            return "invalid_bbox.jpg"

        # Extract crop
        crop = frame[y1_clamped:y2_clamped, x1_clamped:x2_clamped]

        # Verify crop is not empty
        if crop.size == 0 or crop.shape[0] == 0 or crop.shape[1] == 0:
            print(f"  [WARNING] Empty crop for track {vehicle_id} frame {frame_id} - skipping evidence")
            return "empty_crop.jpg"

        # Save as JPEG
        filename = f"viol_track{vehicle_id}_frame{frame_id:05d}.jpg"
        filepath = os.path.join(self.evidence_dir, filename)

        try:
            success = cv2.imwrite(filepath, crop, [cv2.IMWRITE_JPEG_QUALITY, 90])
            if not success:
                print(f"  [WARNING] Failed to write {filename}")
                return "write_failed.jpg"
            return filepath
        except Exception as e:
            print(f"  [ERROR] Exception saving evidence: {e}")
            return "exception.jpg"

    def get_violation_count(self) -> int:
        cur = self.conn.cursor()
        cur.execute("SELECT COUNT(*) FROM Violation")
        return cur.fetchone()[0]

    def close(self):
        self.conn.close()

print("[✓] ViolationLogger with ROBUST boundary checking defined")

[✓] ViolationLogger with ROBUST boundary checking defined


## 16. Emergency Vehicle System

Main integration class. Processes one frame at a time: detect → classify → track → compliance check → log → annotate.


In [16]:
class EmergencyVehicleSystem:
    """End-to-end emergency vehicle compliance monitoring system."""

    ALERT_COLOR = (0, 0, 255)    # Red
    NORMAL_COLOR = (0, 200, 0)   # Green

    def __init__(self, det_weights, yolo_cls_weights, resnet_weights,
                 det_conf=0.3, proximity_r=180, reaction_w=30, yield_threshold=40,
                 camera_type="dashcam", db_path="violations.db", evidence_dir="evidence_frames"):

        print("[INFO] Initializing Emergency Vehicle System...")

        self.detector = YOLO(det_weights)
        print("[INFO]   YOLOv8 detection loaded")

        self.classifier = EnsembleClassifier(yolo_cls_weights, resnet_weights)
        print("[INFO]   Ensemble classifier loaded")

        self.tracker = DeepSORTTracker(max_age=30)
        print("[INFO]   DeepSORT tracker initialized")

        self.compliance = ComplianceEngine(proximity_r, reaction_w, yield_threshold, camera_type)

        self.logger = ViolationLogger(db_path, evidence_dir)

        self.det_conf = det_conf

        print("[✓] System initialization complete\n")

    def _get_box_color(self, label):
        """Get bounding box color based on vehicle type."""
        return self.ALERT_COLOR if label == "emergency" else self.NORMAL_COLOR

    def process_frame(self, frame, frame_id=0):
        """Process single frame through full pipeline."""

        H, W = frame.shape[:2]

        # 1. Detection
        results = self.detector(frame, conf=self.det_conf, verbose=False)[0]
        raw_detections = []

        if results.boxes is not None:
            for box in results.boxes:
                x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].cpu().numpy()]
                det_conf = float(box.conf[0])

                # 2. Classification
                crop = frame[max(0, y1):min(H, y2), max(0, x1):min(W, x2)]
                if crop.size == 0:
                    continue

                crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                crop_pil = Image.fromarray(crop_rgb)

                label, cls_conf, _ = self.classifier.predict(crop_pil)
                final_conf = det_conf * cls_conf

                raw_detections.append([x1, y1, x2, y2, final_conf, label])

        # 3. Tracking
        tracks = self.tracker.update(raw_detections, frame)

        # 4. Compliance
        violations = self.compliance.check(tracks, frame_id, frame)
        self.logger.log_many(violations, frame)

        # 5. Annotation
        annotated = frame.copy()
        alert_ids = []

        for tid, x1, y1, x2, y2, label in tracks:
            color = self._get_box_color(label)

            cv2.rectangle(annotated, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
            cv2.putText(annotated, f"ID:{tid} {label}", (int(x1), max(int(y1) - 8, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            if label == "emergency":
                alert_ids.append(tid)

        # Violation overlay
        for v in violations:
            bx1, by1, bx2, by2 = v["bbox"]
            cv2.putText(annotated, "BLOCKING", (bx1, by2 + 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        # Status overlay
        if alert_ids:
            cv2.putText(annotated, f"EMERGENCY VEHICLE DETECTED (IDs: {alert_ids})",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, self.ALERT_COLOR, 2)

        cv2.putText(annotated, f"Violations: {self.logger.get_violation_count()}",
                    (10, annotated.shape[0] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

        return annotated, alert_ids

    def process_video(self, video_path, output_path="output.mp4"):
        """Process entire video."""

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise IOError(f"Cannot open: {video_path}")

        W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

        writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

        frame_id = 0
        alert_frames = 0

        print(f"[INFO] Processing: {video_path}")
        print(f"[INFO] Resolution: {W}×{H} @ {fps:.1f} FPS")

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            annotated, alerts = self.process_frame(frame, frame_id)

            if alerts:
                alert_frames += 1

            writer.write(annotated)
            frame_id += 1

            if frame_id % 100 == 0:
                print(f"  Frame {frame_id} | EV alerts: {alert_frames} | Violations: {self.logger.get_violation_count()}")

        cap.release()
        writer.release()

        print(f"\n{'='*60}")
        print(f"  PROCESSING COMPLETE")
        print(f"{'='*60}")
        print(f"  Total frames:    {frame_id}")
        print(f"  EV detections:   {alert_frames}")
        print(f"  Violations:      {self.logger.get_violation_count()}")
        print(f"  Output video:    {output_path}")
        print(f"  Database:        {self.logger.db_path}")
        print(f"  Evidence frames: {self.logger.evidence_dir}/")

        self.logger.close()

print("[✓] EmergencyVehicleSystem ready")

[✓] EmergencyVehicleSystem ready


## 17. Training Entry Point

`main_train()` runs the complete training workflow: dataset download → K-Fold CV → test evaluation → save weights and plots. Expected runtime: ~35–45 min on GPU.


In [17]:
def main_train(k: int = 5, yolo_epochs: int = 8,
               resnet_epochs: int = 4, batch: int = 32):
    """
    Runs the full training pipeline.

    Args:
        k: Number of K-Fold splits (default 5)
        yolo_epochs: Training epochs for YOLOv8-cls
        resnet_epochs: Training epochs for ResNet18
        batch: Batch size (reduce if GPU OOM)

    Returns:
        dict: CV results with best fold info
    """
    print("=" * 60)
    print("  EMERGENCY VEHICLE CLASSIFICATION — TRAINING")
    print(f"  YOLOv8-cls + ResNet18 Ensemble  |  K={k}-Fold CV")
    print("=" * 60)

    # Step 1: Download dataset
    dataset_path = download_dataset()

    # Step 2: Collect all samples
    image_paths, labels = collect_all_samples(dataset_path)

    # Step 3: Run K-Fold CV
    cv_info = run_kfold_cv(
        image_paths,
        labels,
        k=k,
        yolo_epochs=yolo_epochs,
        resnet_epochs=resnet_epochs,
        batch=batch,
    )

    # Step 4: Evaluate on held-out test set
    evaluate_on_test_fold(cv_info, image_paths, labels)

    # Step 5: Save summary
    summary = {
        "k": k,
        "mean_acc": cv_info["mean_acc"],
        "std_acc": cv_info["std_acc"],
        "best_fold": cv_info["best"]["fold"],
        "best_val_acc": cv_info["best"]["ensemble_val_acc"],
    }
    Path("cv_summary.json").write_text(json.dumps(summary, indent=2))

    print(f"\n{'='*60}")
    print(f"  TRAINING PIPELINE COMPLETE")
    print(f"{'='*60}")
    print(f"  CV summary saved to: cv_summary.json")
    print(f"  Mean CV accuracy: {cv_info['mean_acc']:.4f} ± {cv_info['std_acc']:.4f}")
    print(f"  Best fold: {cv_info['best']['fold']} "
          f"(val_acc={cv_info['best']['ensemble_val_acc']:.4f})")

    return cv_info

print("[✓] Training pipeline function defined")

[✓] Training pipeline function defined


## 18. Inference Pipeline

`run_on_video()` runs the full trained pipeline on any MP4 and writes an annotated output video with all violations logged to the database.


In [23]:
def run_on_video(det_weights, yolo_cls_weights, resnet_weights, video_path,
                 output_path=None, camera_type="dashcam", db_path=None, evidence_dir=None, **kwargs):
    """
    Process video with trained models.

    Args:
        det_weights: Path to YOLOv8 detection weights
        yolo_cls_weights: Path to YOLOv8 classification weights
        resnet_weights: Path to ResNet18 weights
        video_path: Input video path
        output_path: Output video path
        camera_type: "dashcam" or "overhead"
        db_path: SQLite database path
        evidence_dir: Evidence frames directory
        **kwargs: Additional parameters (proximity_r, reaction_w, yield_threshold, etc.)
    """

    if output_path is None:
        output_path = str(VIDEO_OUT_DIR / "output_annotated.mp4")
    if db_path is None:
        db_path = str(DB_DEMO_PATH)
    if evidence_dir is None:
        evidence_dir = str(EVIDENCE_DIR)

    # Clear existing database
    if Path(db_path).exists():
        Path(db_path).unlink()
        print(f"[INFO] Cleared existing database: {db_path}")

    system = EmergencyVehicleSystem(
    det_weights="yolov8n.pt",
    yolo_cls_weights="/content/best.pt",
    resnet_weights="/content/resnet18_fold3.pt",
    camera_type="dashcam",  # ← Dashcam mode
    proximity_r=180,
    reaction_w=30,
    yield_threshold=40
    )

    '''system = EmergencyVehicleSystem(
    det_weights="yolov8n.pt",
    yolo_cls_weights="/content/best.pt",
    resnet_weights="/content/resnet18_fold3.pt",
    camera_type="overhead",  # ← Overhead mode
    proximity_r=180,
    reaction_w=30,
    yield_threshold=40
    )'''

    system.process_video(video_path=video_path, output_path=output_path)

print("[✓] run_on_video() function ready")

[✓] run_on_video() function ready


## 19. Quick Start

**Option 1** — Full training pipeline (~35–45 min, GPU recommended). Requires Kaggle credentials (`~/.kaggle/kaggle.json`).

**Option 2** — Video inference using pre-trained weights.


In [24]:
# OPTION 1: FULL TRAINING PIPELINE

'''cv_info = main_train(
k=5,              # Number of K-Fold splits
yolo_epochs=8,    # YOLOv8 training epochs per fold
resnet_epochs=4,  # ResNet18 training epochs per fold
batch=32          # Reduce to 16 if GPU runs out of memory
)'''


run_on_video(
    det_weights="yolov8n.pt",
    yolo_cls_weights="/content/best.pt",
    resnet_weights="/content/resnet18_fold3.pt",
    video_path="/content/Input02.mp4",
    camera_type="overhead",
    proximity_r=180,
    reaction_w=30,
    yield_threshold=40
)

'''run_on_video(
    det_weights="yolov8n.pt",
    yolo_cls_weights="/content/best.pt",
    resnet_weights="/content/resnet18_fold3.pt",
    video_path="/content/vid3.mp4",
)'''

# # Outputs saved automatically to:
# #   outputs/videos/output_annotated.mp4
# #   outputs/database/violations_demo.db
# #   outputs/evidence_frames/*.jpg

[INFO] Initializing Emergency Vehicle System...
[INFO]   YOLOv8 detection loaded
[INFO]   Ensemble classifier loaded
[INFO]   DeepSORT tracker initialized
[INFO] Compliance Engine initialized:
  Camera type: dashcam
  Proximity radius: 180px
  Reaction window: 30 frames
  Yield threshold: 40px
[INFO] Violation database: violations.db
[INFO] Evidence directory: evidence_frames/
[✓] System initialization complete

[INFO] Processing: /content/Input02.mp4
[INFO] Resolution: 1280×720 @ 30.0 FPS
  Frame 100 | EV alerts: 70 | Violations: 0
  Frame 200 | EV alerts: 170 | Violations: 0
  [VIOLATION] Frame 00293 | Track 88 | Dist: 72.6px | Yield: 9.7px | BLOCKING (stationary)
  Frame 300 | EV alerts: 270 | Violations: 1
  [VIOLATION] Frame 00322 | Track 50 | Dist: 37.7px | Yield: -44.7px | BLOCKING (approaching)
  [VIOLATION] Frame 00354 | Track 141 | Dist: 73.8px | Yield: 22.8px | BLOCKING (slow yield)
  [VIOLATION] Frame 00357 | Track 141 | Dist: 77.2px | Yield: 39.4px | BLOCKING (slow yield)


In [25]:
from google.colab import files
import shutil

# Download database
files.download('/content/violations.db')

# Zip and download evidence
shutil.make_archive('evidence', 'zip', '/content/evidence_frames')
files.download('evidence.zip')

print("✅ All files downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ All files downloaded!
